In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
data = pd.read_csv("financial_transaction.csv")
data.head()

,Transaction_Text,Label
0,IKEA furniture shopping | Ref:37c9a05e | Amoun...,Shopping
1,MakeMyTrip hotel booking | Ref:7fb0c9d1 | Amou...,Travel
2,Uber ride payment | Ref:f3653a0a | Amount: INR...,Travel
3,Flipkart electronics purchase | Ref:ab027ce0 |...,Shopping
4,HDFC home loan EMI | Ref:44608adc | Amount: IN...,EMI


In [3]:
X = data.drop('Label' , axis=1)
y = data['Label']

# Model 1:-
It categories the raw bank statement csv file into lables:-

In [14]:
import re
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Optional
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.utils.validation import check_is_fitted


@dataclass
class TxnPrediction:
    cleaned_text: str
    predicted_category: str
    confidence: float
    needs_review: bool
    top_k: List[Tuple[str, float]]


class ProductionTransactionClassifier:
    CATEGORIES = [
        "Salary/Income", "Loan EMI", "Investments", "Credit Card Payment",
        "Subscriptions", "Groceries", "Food & Dining", "Travel & Fuel",
        "Entertainment", "Shopping", "Bills & Utilities",
        "Personal Transfer", "Uncategorized"
    ]

    def __init__(self, confidence_threshold: float = 0.72, random_state: int = 42):
        self.confidence_threshold = confidence_threshold
        self.random_state = random_state
        self.model: Optional[Pipeline] = None

    @staticmethod
    def clean_text(raw_text: str) -> str:
        text = str(raw_text).upper().strip()

        text = re.sub(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", " ", text)
        text = text.replace("@", " AT ")
        text = re.sub(r"[/_.\-:|]+", " ", text)

        banking_noise = (
            r"\b(?:UPI|IMPS|NEFT|RTGS|POS|ATM|UTR|RRN|TXN|TRANSACTION|"
            r"REFERENCE|REF|INFO|MOBILE|TRANSFER|PAYMENT ID|ID NO)\b"
        )
        text = re.sub(banking_noise, " ", text)

        text = re.sub(r"\b\d{6,}\b", " ", text)
        text = re.sub(r"([A-Z]+)\d+([A-Z]*)", r" \1 \2 ", text)
        text = re.sub(r"\d+([A-Z]+)", r" \1 ", text)

        text = re.sub(r"[^A-Z\s&]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    @staticmethod
    def merchant_group_key(raw_text: str) -> str:
        text = ProductionTransactionClassifier.clean_text(raw_text)
        stop = {
            "TO", "FROM", "BY", "AT", "ONLINE", "INDIA", "PAY", "PAYMENT",
            "TRANSFER", "UPI", "IMPS", "BANK", "MOBILE", "SERVICE"
        }
        tokens = [t for t in text.split() if t not in stop and len(t) > 2]
        return " ".join(tokens[:4]) if tokens else "UNKNOWN"

    @staticmethod
    def weak_label(raw_text: str) -> str:
        t = str(raw_text).upper()

        if re.search(r"\b(?:SALARY|SAL|PAYROLL|PAY\s?ROLL|CMS\s*/?\s*SAL|ACH\s*/?\s*SAL|SAL\s*CREDIT)\b", t):
            return "Salary/Income"
        if re.search(r"\b(?:HOME LOAN|LOAN EMI|EMI|ECS EMI|NACH EMI)\b", t):
            return "Loan EMI"
        if re.search(r"\b(?:MUTUAL FUND|SIP|FIXED DEPOSIT|FD BOOKING|GROWW|ZERODHA|COIN|KUVERA)\b", t):
            return "Investments"
        if re.search(r"\b(?:CREDIT CARD|CARD PAYMENT|CARD MONTHLY DUE|CC PAYMENT|CRED)\b", t):
            return "Credit Card Payment"
        if re.search(r"\b(?:NETFLIX|SPOTIFY|YOUTUBE PREMIUM|ICLOUD|APPLE COM BILL|HOTSTAR)\b", t):
            return "Subscriptions"
        if re.search(r"\b(?:SWIGGY INSTAMART|BLINKIT|BIGBASKET|KIRANA|GROCERY|DMART|JIOMART)\b", t):
            return "Groceries"
        if re.search(r"\b(?:ZOMATO|SWIGGY|DOMINOS|MCDONALDS|BURGER KING|STARBUCKS|CAFE|RESTAURANT)\b", t):
            return "Food & Dining"
        if re.search(r"\b(?:UBER|OLA|RAPIDO|PETROL|FUEL|HPCL|IOCL|BPCL|MAKEMYTRIP|IRCTC|AIR INDIA|INDIGO)\b", t):
            return "Travel & Fuel"
        if re.search(r"\b(?:PVR|INOX|CINEMA|MOVIE|BOOKMYSHOW)\b", t):
            return "Entertainment"
        if re.search(r"\b(?:AMAZON(?!\s*PAY)|MYNTRA|ZARA|DECATHLON|AJIO|FLIPKART|SHOPPING)\b", t):
            return "Shopping"
        if re.search(r"\b(?:JIO|AIRTEL|VI |VODAFONE|ELECTRICITY|WATER|GAS|BROADBAND|FASTAG|INSURANCE|LIC)\b", t):
            return "Bills & Utilities"
        if re.search(r"\b(?:TRANSFER TO|SENT TO|P2P|PERSONAL TRANSFER|FRIEND|SELF TRANSFER)\b", t):
            return "Personal Transfer"

        return "Uncategorized"

    @staticmethod
    def add_features(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        desc = out["Description"].fillna("").astype(str)
        up = desc.str.upper()

        out["cleaned_text"] = desc.apply(ProductionTransactionClassifier.clean_text)
        out["amount"] = pd.to_numeric(out["Amount"], errors="coerce").fillna(0.0)
        out["abs_amount"] = out["amount"].abs()
        out["log_amount"] = np.log1p(out["abs_amount"])

        out["is_credit"] = up.str.contains(r"\b(?:CREDITED|SAL CREDIT|BY CASH DEPOSIT|ACH CR|NEFT CR)\b", regex=True).astype(int)
        out["has_emi"] = up.str.contains(r"\b(?:EMI|LOAN|ECS|NACH)\b", regex=True).astype(int)
        out["has_investment"] = up.str.contains(r"\b(?:SIP|MUTUAL FUND|FIXED DEPOSIT|FD|GROWW|ZERODHA|COIN|KUVERA)\b", regex=True).astype(int)
        out["has_transfer"] = up.str.contains(r"\b(?:TRANSFER|SENT TO|P2P|SELF TRANSFER|IMPS OUT|IFT OUT)\b", regex=True).astype(int)
        out["has_upi"] = up.str.contains(r"\bUPI\b|@", regex=True).astype(int)
        out["has_bill_hint"] = up.str.contains(r"\b(?:ELECTRICITY|WATER|GAS|BROADBAND|RECHARGE|FASTAG|INSURANCE|LIC)\b", regex=True).astype(int)
        out["has_travel_hint"] = up.str.contains(r"\b(?:UBER|OLA|RAPIDO|PETROL|FUEL|IRCTC|MAKEMYTRIP|INDIGO|AIR)\b", regex=True).astype(int)
        out["has_food_hint"] = up.str.contains(r"\b(?:ZOMATO|SWIGGY|RESTAURANT|CAFE|EATERY)\b", regex=True).astype(int)
        out["has_grocery_hint"] = up.str.contains(r"\b(?:KIRANA|GROCERY|INSTAMART|BLINKIT|BIGBASKET|JIOMART)\b", regex=True).astype(int)
        out["has_cc_hint"] = up.str.contains(r"\b(?:CREDIT CARD|CARD PAYMENT|CC PAYMENT|CRED)\b", regex=True).astype(int)
        out["has_salary_hint"] = up.str.contains(r"\b(?:SALARY|SAL|PAYROLL|CMS/?SAL|ACH_SAL|ACH SAL)\b", regex=True).astype(int)
        out["has_amazon_pay"] = up.str.contains(r"\bAMAZON PAY\b|\bAMAZONPAY\b", regex=True).astype(int)
        out["has_paytm"] = up.str.contains(r"\bPAYTM\b", regex=True).astype(int)
        out["has_phonepe"] = up.str.contains(r"\bPHONEPE\b", regex=True).astype(int)
        out["merchant_key"] = desc.apply(ProductionTransactionClassifier.merchant_group_key)

        return out

    def build_pipeline(self) -> Pipeline:
        text_features = FeatureUnion([
            ("word_tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.98,
                sublinear_tf=True,
                strip_accents="unicode"
            )),
            ("char_tfidf", TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                min_df=2,
                sublinear_tf=True
            ))
        ])

        numeric_cols = [
            "amount", "abs_amount", "log_amount", "is_credit", "has_emi",
            "has_investment", "has_transfer", "has_upi", "has_bill_hint",
            "has_travel_hint", "has_food_hint", "has_grocery_hint",
            "has_cc_hint", "has_salary_hint", "has_amazon_pay",
            "has_paytm", "has_phonepe"
        ]

        preprocessor = ColumnTransformer([
            ("text", text_features, "cleaned_text"),
            ("merchant", TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True
            ), "merchant_key"),
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("scaler", StandardScaler())
            ]), numeric_cols)
        ])

        return Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(
                solver="lbfgs",
                max_iter=1000,
                C=1.0,
                class_weight="balanced",
                random_state=self.random_state
            ))
        ])

    def fit(self, df: pd.DataFrame, y: pd.Series):
        X = self.add_features(df)
        self.model = self.build_pipeline()
        self.model.fit(X, y)
        return self

    def predict_one(self, description: str, amount: float) -> TxnPrediction:
        check_is_fitted(self.model)
        X = pd.DataFrame([{"Description": description, "Amount": amount}])
        Xf = self.add_features(X)

        probs = self.model.predict_proba(Xf)[0]
        classes = self.model.classes_
        best_idx = int(np.argmax(probs))

        predicted = classes[best_idx]
        confidence = float(probs[best_idx])
        top_idx = np.argsort(probs)[::-1][:3]
        top_k = [(classes[i], float(probs[i])) for i in top_idx]

        return TxnPrediction(
            cleaned_text=Xf.iloc[0]["cleaned_text"],
            predicted_category=predicted,
            confidence=confidence,
            needs_review=confidence < self.confidence_threshold,
            top_k=top_k
        )

In [15]:
import pandas as pd
import numpy as np

# ==========================================
# 1. MAKE SURE YOUR REFACTORED CLASS DEFINITION IS HERE
# ==========================================
# (Ensure your updated build_pipeline has solver="lbfgs" and no multi_class argument)

if __name__ == "__main__":
    # 1. Load your local transaction file
    # Ensure your file has "Description" and "Amount" columns
    input_filename = "testing.csv"
    output_filename = "predicted_transactions.csv"
    
    print(f"--- Loading data from {input_filename} ---")
    df = pd.read_csv(input_filename)
    
    # 2. Initialize your production classifier
    classifier = ProductionTransactionClassifier(confidence_threshold=0.72)
    
    # 3. Bootstrap training labels using the rule-based weak_label logic
    print("--- Simulating labels via keyword logic ---")
    df["Target"] = df["Description"].apply(classifier.weak_label)
    
    # 4. Feature engineering and preparation
    fe_data = classifier.add_features(df)
    
    # 5. Initialize and train the machine learning pipeline
    print("--- Training the Machine Learning Pipeline ---")
    classifier.model = classifier.build_pipeline()
    classifier.model.fit(fe_data, fe_data["Target"])
    
    # 6. Extract classification probability distribution across all categories
    print("--- Computing model predictions & confidence metrics ---")
    probabilities = classifier.model.predict_proba(fe_data)
    classes = classifier.model.classes_
    
    # Arrays to store tracking attributes row-by-row
    pred_categories = []
    confidences = []
    needs_review_flags = []
    
    for prob_dist in probabilities:
        # Match class names to confidence intervals and sort high -> low
        class_probs = sorted(zip(classes, prob_dist), key=lambda x: x[1], reverse=True)
        best_class, best_prob = class_probs[0]
        
        pred_categories.append(best_class)
        confidences.append(best_prob)
        # Tag rows where the algorithm is guessing or uncertain
        needs_review_flags.append(best_prob < classifier.confidence_threshold)
        
    # 7. Map metrics back onto original structure columns
    df["Cleaned_Text"] = fe_data["cleaned_text"]
    df["Predicted_Category"] = pred_categories
    df["Confidence_Score"] = confidences
    df["Needs_Review"] = needs_review_flags
    
    # Remove the internal 'Target' temporary calculation column before saving
    final_export_df = df.drop(columns=["Target"])
    
    # 8. Save output directly to file disk
    final_export_df.to_csv(output_filename, index=False)
    print(f"--- Process Finished successfully! Output saved to: '{output_filename}' ---")

--- Loading data from testing.csv ---
--- Simulating labels via keyword logic ---
--- Training the Machine Learning Pipeline ---
--- Computing model predictions & confidence metrics ---
--- Process Finished successfully! Output saved to: 'predicted_transactions.csv' ---
